# 01 · Exploring the PKLot data

**Goal:** understand the dataset *before* building any model.

Questions this notebook answers:
1. How is the annotation file organized?
2. How is one parking space described?
3. How many spaces are empty vs. occupied? (Is it balanced?)
4. Which camera / parking lot does each image come from?
5. Is the data clean? (missing labels, overlapping boxes)

In [ ]:
import json
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# Roboflow split the data into 3 folders. Each one has its own annotation file.
SPLITS = ["train", "valid", "test"]

data = {}
for split in SPLITS:
    with open(f"{split}/_annotations.coco.json") as f:
        data[split] = json.load(f)

print("Loaded:", list(data.keys()))

## 1. How the file is organized

The labels use the **COCO format**, a standard used by many vision datasets. It's one big dictionary.
The three parts that matter:

- `images`: one entry per photo (file name, width, height, id)
- `annotations`: one entry per **parking space** in a photo (where the box is and its label)
- `categories`: what each label number means

Annotations point back to their photo using `image_id`. That's how you know which spaces belong to which picture.

In [ ]:
train = data["train"]

print("Top-level keys:", list(train.keys()))
print("Images:        ", len(train["images"]))
print("Annotations:   ", len(train["annotations"]))
print()
for cat in train["categories"]:
    print(cat)

Category `0` ("spaces") is just a parent group Roboflow adds. It's never actually used as a label.
The two real classes are **1 = empty** and **2 = occupied**.

## 2. What one parking space looks like

In [ ]:
first_image = train["images"][0]
print("Image:", first_image)
print()

# All annotations that belong to that image
spaces = [a for a in train["annotations"] if a["image_id"] == first_image["id"]]
print("Spaces in this image:", len(spaces))
print("First space:", spaces[0])

`bbox` is **[x, y, width, height]** in pixels:
- `x, y` = the box's **top-left corner** (0,0 is the top-left of the image)
- `width, height` = how big the box is

So `[278, 185, 46, 45]` is a 46×45 pixel box starting 278 px from the left and 185 px from the top.
Every image is 1280×720.

## 3. Class balance: empty vs. occupied

If 95% of spaces were empty, a "model" that always guesses *empty* would score 95% while being useless.
So we check the balance before trusting any accuracy number later.

In [ ]:
NAMES = {1: "empty", 2: "occupied"}

for split in SPLITS:
    counts = Counter(a["category_id"] for a in data[split]["annotations"])
    total = sum(counts.values())
    summary = ", ".join(f"{NAMES[c]}: {n:,} ({n / total:.1%})" for c, n in sorted(counts.items()))
    print(f"{split:5s} -> {summary}")

Roughly 50/50 in every split, so plain accuracy is a fair metric here. That's not something to assume; we checked.

## 4. Which camera does each image come from?

PKLot was filmed by **3 cameras at 2 parking lots**: PUCPR, UFPR04 and UFPR05.
Roboflow renamed the files, so the lot name is gone. But each camera always sees the same fixed set of spaces,
so **the number of labeled spaces per image identifies the camera**.

In [ ]:
spaces_per_image = Counter(a["image_id"] for a in train["annotations"])
print("Most common space counts (spaces per image -> number of images):")
print(Counter(spaces_per_image.values()).most_common(5))

In [ ]:
# The three counts map to the three cameras described in the PKLot paper.
CAMERA_BY_COUNT = {100: "PUCPR", 28: "UFPR04", 40: "UFPR05"}

def camera_of_each_image(split_data):
    counts = Counter(a["image_id"] for a in split_data["annotations"])
    return {img["id"]: CAMERA_BY_COUNT.get(counts.get(img["id"], 0), "unknown")
            for img in split_data["images"]}

print(f"{'split':6s} {'PUCPR':>7s} {'UFPR04':>7s} {'UFPR05':>7s} {'unknown':>8s}")
for split in SPLITS:
    cams = Counter(camera_of_each_image(data[split]).values())
    print(f"{split:6s} {cams['PUCPR']:7d} {cams['UFPR04']:7d} {cams['UFPR05']:7d} {cams['unknown']:8d}")

### ⚠️ Important finding

**Every camera appears in train, valid *and* test.** Roboflow shuffled the images randomly.
On top of that, the cameras took a photo every 5 minutes, so the test set is full of near-copies of training photos.

That means a model tested on Roboflow's test set has basically **already seen those exact parking spots**,
so its score will look great but won't tell us if it works on a new lot.

**Decision:** ignore Roboflow's split. Make our own:
- **train** on UFPR04 + UFPR05 (the UFPR lot)
- **test** on PUCPR (a different lot the model has never seen)

## 5. Data problems

### Images with no labels

In [ ]:
for split in SPLITS:
    labeled = {a["image_id"] for a in data[split]["annotations"]}
    unlabeled = [img for img in data[split]["images"] if img["id"] not in labeled]
    print(f"{split:5s}: {len(unlabeled)} images have no labeled spaces")

These can't be used for training (we don't know which spaces are empty), so we'll skip them.

### Look at the actual images

Green = empty, red = occupied. One example from each camera.

In [ ]:
def show_image_with_boxes(split, image_entry, title=""):
    img = Image.open(f"{split}/{image_entry['file_name']}")
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.imshow(img)
    for a in data[split]["annotations"]:
        if a["image_id"] != image_entry["id"]:
            continue
        x, y, w, h = a["bbox"]
        color = "lime" if a["category_id"] == 1 else "red"
        ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=1.2))
    ax.set_title(title)
    ax.axis("off")
    plt.show()

cams = camera_of_each_image(train)
for camera in ["PUCPR", "UFPR04", "UFPR05"]:
    example = next(img for img in train["images"] if cams[img["id"]] == camera)
    show_image_with_boxes("train", example, title=f"{camera}  ({example['file_name'][:19]})")

### Overlapping boxes

The original PKLot labels are **rotated** rectangles that follow each space's angle.
Roboflow replaced each one with a normal, straight box around it. When spaces are angled and packed tightly,
those straight boxes overlap, so a box can include part of the **neighboring car**.
How common is that?

Neighboring boxes almost always touch a little, so "does it overlap at all?" isn't useful.
Instead we measure **how much** of each box is covered by the biggest neighboring box.

In [ ]:
def intersection_area(a, b):
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    width = min(ax + aw, bx + bw) - max(ax, bx)
    height = min(ay + ah, by + bh) - max(ay, by)
    return max(width, 0) * max(height, 0)

def worst_neighbor_overlaps(split_data, camera, cams, max_images=50):
    # For every box: the largest share of its area that a single other box covers.
    by_image = {}
    for a in split_data["annotations"]:
        by_image.setdefault(a["image_id"], []).append(a["bbox"])
    image_ids = [i for i in by_image if cams[i] == camera][:max_images]
    shares = []
    for i in image_ids:
        boxes = by_image[i]
        for j, box in enumerate(boxes):
            area = box[2] * box[3]
            worst = max(intersection_area(box, other) for k, other in enumerate(boxes) if k != j)
            shares.append(worst / area)
    return shares

for camera in ["PUCPR", "UFPR04", "UFPR05"]:
    shares = sorted(worst_neighbor_overlaps(train, camera, cams))
    median = shares[len(shares) // 2]
    heavy = sum(s > 0.25 for s in shares) / len(shares)
    print(f"{camera:7s}: median overlap {median:.0%}, boxes over 25% covered by a neighbor: {heavy:.0%}")

## Summary

- **Format:** COCO. Each space = a straight box `[x, y, w, h]` + label (1 empty, 2 occupied).
- **Balance:** about 50/50 empty vs. occupied, so accuracy is a fair metric.
- **Cameras:** identifiable by spaces per image (100 = PUCPR, 28 = UFPR04, 40 = UFPR05).
- **Problem 1:** Roboflow's split mixes all cameras and near-duplicate frames → its test score would be misleading.
  We'll split by lot instead: train on UFPR, test on PUCPR.
- **Problem 2:** some images have no labels → skip them.
- **Problem 3:** straight boxes can overlap neighbors → crops may include part of the next car.

**Next (notebook 02):** cut each labeled space out of its image into a small square "patch",
which becomes the input for an empty-vs-occupied classifier.